In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import CRS, Transformer
import folium
import json
import webbrowser
from haversine import haversine, Unit
import math

grid定义

In [ ]:
class Grid:
    def __init__(self, lon, lat, gridId, maxLon, maxLat, minLon, minLat ):
        self.lon = lon
        self.lat = lat
        self.gridId = gridId
        self.demand = 0     #网格配送需求
        self.stationCost = 0    #建配送站费用
        self.height = 0     #建筑最大高度
        self.weatherCost = 0    #天气消耗
        self.isStation = 0      #是否为配送站
        self.maxLon = maxLon
        self.maxLat = maxLat
        self.minLon = minLon
        self.minLat = minLat

    def __repr__(self):
        return f"grid({self.gridId})"
    def setDemand(self, demand):
        self.demand = demand
    def setStationCost(self, stationCost):
        self.stationCost = stationCost
    def setHeight(self, height):
        self.height = height
    def setWeatherCost(self, weatherCost):
        self.weatherCost = weatherCost
    def setStation(self):
        self.isStation = 1

划分方格

In [ ]:
def createYangpuGrid(lonMin = 121.481, lonMax = 121.562, latMin = 31.25, latMax = 31.348, gridSize = 200):
    # 坐标系转换
    wgs84 = CRS("EPSG:4326")
    cgcs_sh = CRS("EPSG:32651")
    trans = Transformer.from_crs(wgs84, cgcs_sh, always_xy=True)
    back = Transformer.from_crs(cgcs_sh, wgs84, always_xy=True)

    # 杨浦区边界
    # lonMin, lonMax = 121.46, 121.58   # 经度
    # latMin, latMax = 31.23, 31.31     # 纬度

    # 转平面坐标
    xMin, yMin = trans.transform(lonMin, latMin)
    xMax, yMax = trans.transform(lonMax, latMax)

    #范围内xy坐标
    xs = np.arange(xMin, xMax, gridSize)   # 横向x
    ys = np.arange(yMin, yMax, gridSize)   # 纵向y

    # 生成grid
    grids = []
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            cx = x + gridSize / 2
            cy = y + gridSize / 2
            lon, lat = back.transform(cx, cy)
            maxLon, maxLat = back.transform(x+gridSize, y+gridSize)
            minLon, minLat = back.transform(x, y)
            grid_ = Grid(round(lon, 6), round(lat, 6), f"{i}_{j}", round(maxLon, 6), round(maxLat, 6), round(minLon, 6), round(minLat, 6))
            grids.append(grid_)
    return grids
YangPuGrids = createYangpuGrid()

格子可视化

In [ ]:
def visualizeGrids(grids, gridSize=200, zoomStart = 13, centerLat = 31.27, centerLon = 121.53):
    m = folium.Map(location = [centerLat, centerLon], zoom_start = zoomStart, tiles='OpenStreetMap')

    wgs84 = CRS("EPSG:4326")
    cgcs_sh = CRS("EPSG:32651")
    trans = Transformer.from_crs(wgs84, cgcs_sh, always_xy=True)
    back = Transformer.from_crs(cgcs_sh, wgs84, always_xy=True)

    with open("../data/yangpu.json", 'r', encoding='utf-8') as f:
        yangpuJson = json.load(f)

    folium.GeoJson(
        yangpuJson,
        style_function=lambda feature:{
            'color': 'black',
            'weight': 2,
            'fill': False
        }
    ).add_to(m)

    for g in grids:
        cx, cy = trans.transform(g.lon, g.lat)
        half = gridSize / 2
        corners = [
            (cx - half, cy + half),
            (cx + half, cy + half),
            (cx + half, cy - half),
            (cx - half, cy - half),
        ]
        poly = [back.transform(x, y)[::-1] for x, y in corners]
        folium.Polygon(
            locations=poly,
            color='red',
            weight=1,
            fill=True,
            fillcolor='lightred',
            fillOpacity=0.1,
            popup=f'Grid ID:{g.gridId}'
        ).add_to(m)
        # folium.CircleMarker(
        #     location=[g.lat, g.lon],    #?
        #     radius=1,
        #     color='red',
        #     fill=True,
        #     fillColor='red'
        # )
    return m

mapPic = visualizeGrids(YangPuGrids)
mapPic  # 7,21->19,35

excel表格导出

In [ ]:
gridChart = []
for g in YangPuGrids:
    gridChart.append({
        "gridId": g.gridId,
        "lon": g.lon,
        "lat": g.lat,
        "minLon": g.minLon,
        "maxLon": g.maxLon,
        "minLat": g.minLat,
        "maxLat": g.maxLat
        })
df = pd.DataFrame(gridChart)
df.to_excel("../data/yangpuGrids.xlsx", index = False)

In [20]:
import pandas as pd
import numpy as np

# ===================== 配置路径，直接运行无需修改 =====================
GRID_FILE = "../data/yangpuGridsHeight.xlsx"    # 原始网格模板
POINT_CSV = "../data/test8.csv"                 # 人口点数据
OUTPUT_FILE = "../data/yangpuGridsHeight_with_pop.xlsx"  # 输出带population的完整网格表

# 1. 读取原始网格模板（包含gridId、lon、lat、minLon、maxLon、minLat、maxLat、maxHeight）
grid_df = pd.read_excel(GRID_FILE)
print(f"读取网格模板，总网格数：{len(grid_df)}")

# 2. 读取人口点位数据（longitude, latitude, population）
point_df = pd.read_csv(POINT_CSV)
print(f"读取人口点位，总点数：{len(point_df)}")

# 3. 给每个网格计算中心经纬度，匹配点位所属格子
# 生成网格映射字典 gridId -> (minLon, maxLon, minLat, maxLat)
grid_bounds = {}
for _, row in grid_df.iterrows():
    gid = row["gridId"]
    grid_bounds[gid] = (row["minLon"], row["maxLon"], row["minLat"], row["maxLat"])

# 4. 定义函数：判断一个点落在哪个gridId里
def get_point_grid_id(lon, lat):
    for gid, (minLon, maxLon, minLat, maxLat) in grid_bounds.items():
        if minLon <= lon <= maxLon and minLat <= lat <= maxLat:
            return gid
    return None  # 点不在任何网格内

# 给每个点位匹配gridId
point_df["gridId"] = point_df.apply(lambda r: get_point_grid_id(r["longitude"], r["latitude"]), axis=1)
# 过滤掉不在网格范围内的点
point_valid = point_df[point_df["gridId"].notna()]

# 5. 按网格汇总人口总和
pop_agg = point_valid.groupby("gridId")["population"].sum().reset_index()
pop_agg.rename(columns={"population": "population"}, inplace=True)

# 6. 人口合并到原始网格表，无人口网格填充0
result_df = pd.merge(grid_df, pop_agg, on="gridId", how="left")
result_df["population"] = result_df["population"].fillna(0)

# 7. 输出文件，结构和yangpuGridsHeight.xlsx完全一致，末尾新增population列
result_df.to_excel(OUTPUT_FILE, index=False)
print(f"已生成带人口字段的完整网格文件：{OUTPUT_FILE}")
print("\n前5行预览：")
print(result_df.head())

读取网格模板，总网格数：2160
读取人口点位，总点数：716187
已生成带人口字段的完整网格文件：../data/yangpuGridsHeight_with_pop.xlsx

前5行预览：
  gridId         lon        lat      minLon      maxLon     minLat     maxLat  \
0    0_0  121.482035  31.250914  121.481000  121.483071  31.250000  31.251829   
1    0_1  121.482007  31.252718  121.480971  121.483042  31.251804  31.253633   
2    0_2  121.481978  31.254522  121.480942  121.483013  31.253608  31.255437   
3    0_3  121.481949  31.256326  121.480913  121.482984  31.255412  31.257241   
4    0_4  121.481920  31.258130  121.480884  121.482955  31.257216  31.259044   

   maxHeight  population  
0  52.767900        1543  
1  74.539366        1282  
2  84.975166        1592  
3  82.723566        1168  
4  80.869566         997  
